# 2. Fine-tuning de NLLB avec LoRA sur notre corpus

**Objectif** : adapter NLLB-200-distilled-600M à NOTRE corpus (éwé de 1913 +
segond 1910) avec **LoRA** (Low-Rank Adaptation), pour dépasser la baseline.

## Pourquoi LoRA et pas un fine-tuning complet ?

- Un fine-tuning complet modifierait les **600M de paramètres** → GPU saturé,
  heures d'entraînement, risques d'oubli catastrophique.
- **LoRA** ne modifie que de petits "adaptateurs" (~0,5 % des paramètres)
  ajoutés aux couches d'attention : rapide, léger, et le modèle de base reste
  intact.
- Résultat : ~10-20 min d'entraînement sur un T4 gratuit pour 3 époques.

## Pipeline

1. Charger `train.tsv` (52 512 paires) et `dev.tsv` (6 564 paires) depuis GitHub
2. Tokeniser les paires (langue source + langue cible NLLB)
3. Ajouter les adaptateurs LoRA
4. Entraîner avec `Seq2SeqTrainer` (HuggingFace)
5. Évaluer sur `test.tsv` et **comparer avec la baseline** (notebook 1)

In [ ]:
# Installation (peft = bibliothèque officielle LoRA de HuggingFace)
!pip install -q transformers sacrebleu pandas sentencepiece datasets peft accelerate

print("✅ Dépendances installées")

In [ ]:
# Imports + GPU
import torch
import pandas as pd
import sacrebleu
import numpy as np
from transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,
                          Seq2SeqTrainer, Seq2SeqTrainingArguments,
                          DataCollatorForSeq2Seq)
from datasets import Dataset
from peft import LoraConfig, get_peft_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("🔧 Device :", device)
# Sur Colab : Exécution > Changer le type d'exécution > T4 GPU

In [ ]:
# Chargement train / dev / test depuis le repo GitHub public
BASE = "https://raw.githubusercontent.com/cherif-tg/tg_nlp_toolkit/main/data/processed/v0.3/"

def charger(nom):
    return pd.read_csv(BASE + nom, sep="\t")

train = charger("train.tsv")
dev   = charger("dev.tsv")
test  = charger("test.tsv")
print(f"✅ train={len(train)} dev={len(dev)} test={len(test)}")
print(train.head(2))

In [ ]:
# Chargement du modèle + tokenizer
MODEL_NAME = "facebook/nllb-200-distilled-600M"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

# On gèle le modèle de base : seuls les adaptateurs LoRA seront entraînés
model.config.use_cache = False   # requis par le Trainer pendant l'entraînement
print("✅ Modèle chargé")

In [ ]:
# Préparation des données au format attendu par le Trainer
# Chaque exemple : "input" = phrase source (préfixée par la langue), "labels" = cible

def preparer(df, src_lang, tgt_lang):
    # Le tokenizer NLLB ajoute automatiquement le préfixe de langue si on
    # fixe src_lang AVANT l'encodage, et forced_bos_token_id à la génération.
    sources, cibles = [], []
    for fr, ee in zip(df["fr"], df["ewe"]):
        sources.append(fr)
        cibles.append(ee)
    return sources, cibles

train_src, train_tgt = preparer(train, "fra_Latn", "ewe_Latn")
dev_src, dev_tgt     = preparer(dev, "fra_Latn", "ewe_Latn")

# Encodage : les entrées sont tokenisées avec la langue source,
# les labels avec la langue cible (pad_token = label -100 pour ignorer le padding)
def tokeniser(sources, cibles):
    tokenizer.src_lang = "fra_Latn"
    enc = tokenizer(sources, padding=True, truncation=True, max_length=128, return_tensors="pt")
    tokenizer.src_lang = "ewe_Latn"
    labels = tokenizer(cibles, padding=True, truncation=True, max_length=128, return_tensors="pt")
    enc["labels"] = labels["input_ids"]
    # -100 = tokens ignorés par la loss (padding)
    enc["labels"][enc["labels"] == tokenizer.pad_token_id] = -100
    return enc

train_ds = Dataset.from_dict(tokeniser(train_src, train_tgt))
dev_ds   = Dataset.from_dict(tokeniser(dev_src, dev_tgt))
print(f"✅ Datasets prêts : train {len(train_ds)} exemples, dev {len(dev_ds)}")

In [ ]:
# Configuration LoRA
# On ajoute des adaptateurs sur les projections Q et V de l'attention
# (cible classique pour les modèles seq2seq).
lora_config = LoraConfig(
    r=16,                # rang de la factorisation (plus = plus de capacité)
    lora_alpha=32,       # échelle de mise à jour (souvent 2×r)
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,   # régularisation
    bias="none",
    task_type="SEQ_2_SEQ_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# Attendu : ~0,5 % des paramètres entraînables seulement !

In [ ]:
# Métrique d'évaluation pendant l'entraînement : chrF++ sur le dev set
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    # On décode les prédictions et les labels (en ignorant les -100)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    # SacreBLEU attend une liste de références par phrase
    refs = [[r] for r in decoded_labels]
    chrf = sacrebleu.corpus.chrf(decoded_preds, refs)
    return {"chrF++": chrf.score}

collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
print("✅ Métrique + collator prêts")

In [ ]:
# Configuration de l'entraînement (adaptée à un T4 gratuit)
training_args = Seq2SeqTrainingArguments(
    output_dir="nllb-ewe-lora",
    num_train_epochs=3,          # 3 passages sur le corpus
    per_device_train_batch_size=8,   # 8 paires par lot (T4 ≈ 16 Go)
    per_device_eval_batch_size=8,
    learning_rate=3e-4,
    warmup_steps=200,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",       # évaluer à chaque fin d'époque
    save_strategy="epoch",
    predict_with_generate=True,  # génère de vraies traductions pour la métrique
    generation_max_length=128,
    fp16=True,                   # demi-précision : +rapide sur T4
    report_to="none",
    push_to_hub=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

print("✅ Trainer prêt — lance l'entraînement avec la cellule suivante")

In [ ]:
# 🚀 LANCEMENT DE L'ENTRAÎNEMENT (~10-20 min sur T4)
trainer.train()

print("✅ Entraînement terminé !")

In [ ]:
# Évaluation finale sur le TEST set (jamais vu par le modèle)
# On compare avec la baseline du notebook 1.

def traduire_model(textes, tgt="ewe_Latn", max_len=128, batch_size=16):
    tokenizer.src_lang = "fra_Latn"
    resultats = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        enc = tokenizer(lot, return_tensors="pt", padding=True,
                        truncation=True, max_length=max_len).to(device)
        with torch.no_grad():
            gen = model.generate(**enc,
                                 forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt),
                                 max_new_tokens=max_len, num_beams=4)
        resultats += tokenizer.batch_decode(gen, skip_special_tokens=True)
    return resultats

preds = traduire_model(test["fr"].tolist())
refs  = test["ewe"].tolist()
chrf = sacrebleu.corpus.chrf(preds, [refs])
bleu = sacrebleu.corpus.bleu(preds, [refs])

print("📊 FR → ÉWÉ après fine-tuning LoRA")
print(f"   chrF++ : {chrf.score:.2f}   (baseline zero-shot à comparer)")
print(f"   BLEU   : {bleu.score:.2f}")

for i in range(3):
    print(f"\n--- Exemple {i+1} ---")
    print(f"FR : {test['fr'].iloc[i]}")
    print(f"Réf: {refs[i]}")
    print(f"Préd: {preds[i]}")

In [ ]:
# Sauvegarde du modèle + export vers HuggingFace (optionnel)
# 1) Sauvegarde locale (dossier modèle complet)
model.save_pretrained("nllb-ewe-lora-final")
tokenizer.save_pretrained("nllb-ewe-lora-final")
print("✅ Modèle sauvegardé dans nllb-ewe-lora-final/")

# 2) Export vers ton compte HuggingFace (cheriftenga)
# Décommente et exécute APRÈS t'être connecté :
#   from huggingface_hub import notebook_login
#   notebook_login()   # colle ton token (réglages > Access Tokens)
#
#   from peft import PeftModel
#   model.push_to_hub("cheriftenga/nllb-200-distilled-600M-ewe-lora")
#   tokenizer.push_to_hub("cheriftenga/nllb-200-distilled-600M-ewe-lora")
print("✅ Prêt pour l'export (voir instructions commentées)")

## Lecture des résultats

- Si **chrF++ fine-tune > chrF++ baseline** (notebook 1) : notre corpus apporte
  un vrai gain → le corpus v0.3 est **utile et publiable**.
- Si le gain est faible : vérifier (a) le nombre d'époques, (b) le `r` de LoRA,
  (c) la taille du corpus. Les données restent la contrainte principale en
  low-resource.

**Prochaines étapes** : démo Gradio (P3), publication HuggingFace,
puis traduction manuelle des grilles (10 thèmes) pour couvrir le domaine
santé/administration.